In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%cudf.pandas.profile
### cell 14 ###

# Compute exactly the same "data_sub" as the original, but entirely on the GPU:
data_sub = (
    df.dropna(subset=["year_added"])           # drop any rows with missing year_added (value_counts would skip these)
      .groupby(["type", "year_added"])       # GPU groupby on type & year
      .size()                                   # count entries per (type, year)
      .unstack(level="year_added")            # pivot years into columns
      .fillna(0)                                # fill missing type–year combos with 0
      .loc[["TV Show", "Movie"]]            # ensure the row order matches the original
      .cumsum(axis=0)                           # cumulative sum down the two rows (for stacking)
      .T                                        # transpose → index=year, columns=type
)
